# 🏆 Advanced Elo Ranking System

This notebook implements an Elo ranking workflow with **home advantage**, **margin of victory**, **season regression**, and **parameter tuning**.


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from src.elo import Team, EloCalculator, regress_to_mean


## 1. Sample Match Data

`home_win`: `1` means home win, `0.5` draw, `0` away win.


In [ ]:
matches = pd.DataFrame({
    'date': pd.to_datetime(['2026-01-01','2026-01-08','2026-01-15','2026-01-22','2026-01-29']),
    'home_team': ['A','B','A','C','B'],
    'away_team': ['B','C','C','A','A'],
    'home_points': [30, 20, 15, 28, 24],
    'away_points': [20, 25, 15, 14, 21],
})
matches['home_win'] = np.select(
    [matches.home_points > matches.away_points, matches.home_points == matches.away_points],
    [1.0, 0.5], default=0.0
)
matches


## 2. Initialize Teams and Elo Calculator

In [ ]:
team_names = set(matches.home_team) | set(matches.away_team)
teams = {name: Team(name) for name in team_names}
elo = EloCalculator(home_advantage=50, k=40)


## 3. Process Matches Chronologically

In [ ]:
history = []

for _, fixture in matches.sort_values('date').iterrows():
    home = teams[fixture.home_team]
    away = teams[fixture.away_team]

    result = elo.update(
        home, away,
        home_score=fixture.home_win,
        home_points=fixture.home_points,
        away_points=fixture.away_points,
    )

    history.append({
        'date': fixture.date,
        'home': home.name,
        'away': away.name,
        **result,
    })

pd.DataFrame(history)


## 4. Generate Leaderboard

In [ ]:
leaderboard = pd.DataFrame([
    {'team': name, 'elo_rating': team.elo_rating}
    for name, team in teams.items()
]).sort_values('elo_rating', ascending=False).reset_index(drop=True)
leaderboard.index += 1
leaderboard


## 5. Off-Season Regression

Ratings are moved one-third of the way toward the mean rating of 1500.


In [ ]:
for team in teams.values():
    team.elo_rating = regress_to_mean(team.elo_rating, fraction=1/3)

[(team.name, round(team.elo_rating, 2)) for team in teams.values()]


## 6. Tune K-Factor and Home Advantage

Evaluate combinations using RMSE between predicted home-win probability and actual outcome.


In [ ]:
def evaluate_parameters(df, k, home_advantage):
    names = set(df.home_team) | set(df.away_team)
    trial_teams = {name: Team(name) for name in names}
    trial_elo = EloCalculator(k=k, home_advantage=home_advantage)
    squared_error = 0

    for _, fixture in df.sort_values('date').iterrows():
        home = trial_teams[fixture.home_team]
        away = trial_teams[fixture.away_team]
        prediction = trial_elo.expected_home_score(home.elo_rating, away.elo_rating)
        squared_error += (prediction - fixture.home_win) ** 2
        trial_elo.update(home, away, fixture.home_win, fixture.home_points, fixture.away_points)

    return np.sqrt(squared_error / len(df))

results = []
for home_advantage in range(20, 101, 10):
    for k in range(10, 71, 10):
        rmse = evaluate_parameters(matches, k, home_advantage)
        results.append({'k': k, 'home_advantage': home_advantage, 'rmse': rmse})

tuning = pd.DataFrame(results)
best = tuning.loc[tuning.rmse.idxmin()]
best


## 7. Visualize Parameter Results

In [ ]:
pivot = tuning.pivot(index='k', columns='home_advantage', values='rmse')
plt.figure(figsize=(10, 6))
plt.imshow(pivot.values, aspect='auto')
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel('Home Advantage')
plt.ylabel('K Factor')
plt.title('RMSE for Elo Parameters')
plt.colorbar(label='RMSE')
plt.show()


## 🚀 Next Steps

- Replace sample data with a CSV/API dataset
- Add database persistence
- Track rating history over time
- Add predictions for future fixtures
- Build a FastAPI or Django API
- Create a React dashboard
